## 05 · The Morning After

Every simulated day starts with the battery at 70%, and both operators finish the day around 33–36%. In a one-day simulation, that missing energy is never paid back: tomorrow starts fresh. In a real village, tomorrow starts with whatever today left behind.

This notebook checks whether the AI's 13% fuel saving survives once that borrowed energy is accounted for.

### Part 1: pricing the borrowed energy

The simplest fix is to treat any battery shortfall at the end of the day as a debt, and price it at the cost of refilling it with diesel: 0.3 litres per kWh at $0.60 a litre, plus the 8% lost when charging. That comes to about $0.20 per kWh.

In [1]:
# Run both operators for 20 days, then add the cost of refilling each day's battery shortfall with diesel
from pathlib import Path
import matplotlib.pyplot as plt
from minigrid_mdp.env.minigrid_env import MiniGridEnv
from minigrid_mdp.agents import RuleBasedAgent, DRLAgent
from minigrid_mdp.metrics import run_days, summarise, compare

env = MiniGridEnv()
ai_agent = DRLAgent(env.action_space, model_path=str(Path("..") / "models" / "ppo_fixed_reward.zip"))
rulebook = RuleBasedAgent(env.action_space)

bat, die = env.bat_cfg, env.die_cfg
REFILL_COST_PER_KWH = die["fuel_consumption_l_per_kwh"] * die["fuel_price_usd_per_litre"] / bat["round_trip_efficiency"]

def with_battery_debt(summary):
    shortfall_kwh = (bat["initial_soc"] - summary.end_soc) * bat["capacity_kwh"]
    return summary.assign(adjusted_cost_usd=summary.fuel_cost_usd + shortfall_kwh * REFILL_COST_PER_KWH)

results = {
    "rulebook": with_battery_debt(summarise(run_days(env, rulebook))),
    "ai_fixed_reward": with_battery_debt(summarise(run_days(env, ai_agent))),
}
compare(results)

rulebook       ai_fixed_reward      
                      mean   std            mean   std
diesel_hours_pct     38.96  2.04           25.62  1.53
fuel_cost_usd        33.52  1.61           29.26  2.00
unmet_pct             0.08  0.28            0.06  0.17
end_soc               0.33  0.06            0.36  0.12
reward              -34.38  3.04          -30.16  2.38
adjusted_cost_usd    40.72  2.32           35.93  2.05

**What this shows:** once each day's battery shortfall is priced at the cost of refilling it with diesel, both operators' daily costs rise by about $7. The AI still comes out ahead: $35.93 a day against $40.72, a saving of about 12%. Its advantage was never coming from borrowing more stored energy than the rulebook.

### Part 2: a week without resets

Pricing the debt is an estimate. The stronger test is to stop resetting the battery: run each operator for a full week, so every day starts with whatever the previous day left behind. We run 4 weeks and ignore the first day of each, since it still starts from the artificial 70%.

In [2]:
# Run each operator for 4 continuous weeks, split into days, and summarise days 2–7 of each week
DAYS_PER_WEEK = 7

week_env = MiniGridEnv()
week_env.ep_len = 24 * DAYS_PER_WEEK

def run_weeks(env, agent, n_weeks=4):
    log = run_days(env, agent, n_weeks)
    return log.assign(episode=log.episode * DAYS_PER_WEEK + log.step // 24)

def steady_state_days(log):
    return log[log.episode % DAYS_PER_WEEK != 0]

week_results = {
    "rulebook": summarise(steady_state_days(run_weeks(week_env, rulebook))),
    "ai_fixed_reward": summarise(steady_state_days(run_weeks(week_env, ai_agent))),
}
compare(week_results)

rulebook       ai_fixed_reward       
                     mean   std            mean    std
diesel_hours_pct    47.40  2.40           27.95   2.60
fuel_cost_usd       39.30  1.89           29.95   2.59
unmet_pct            0.31  0.69            5.94   2.30
end_soc              0.34  0.07            0.36   0.12
reward             -42.10  5.94          -77.77  18.94

In [3]:
# Show average unmet demand (kW) by hour of day in the continuous-week runs, for each operator
import pandas as pd

def unmet_by_hour(log):
    return steady_state_days(log).groupby("hour").unmet_load_kw.mean()

pd.DataFrame({
    "rulebook": unmet_by_hour(run_weeks(week_env, rulebook)),
    "ai_fixed_reward": unmet_by_hour(run_weeks(week_env, ai_agent)),
}).round(1)

,rulebook,ai_fixed_reward
hour,,
0,0.0,0.0
1,0.0,0.0
2,0.0,0.2
3,0.0,2.8
4,0.0,5.5
5,0.0,6.3
6,0.0,5.3
7,0.0,2.5
8,0.0,0.0


**What this shows:** when the battery carries over from day to day, the picture flips. The rulebook adapts: it runs diesel more often (47% of hours), costs about $39 a day, and still serves almost all demand. The AI stays cheaper, but leaves about 6% of demand unserved, and on the full score, which counts both fuel and reliability, it now does much worse than the rulebook.

## Summary

- Pricing each day's battery shortfall as a debt trims the AI's advantage only slightly, to about 12%.
- But once days stop resetting, the AI's savings come at the cost of 6% unserved demand, all before dawn. On the full score it loses clearly to the rulebook.
- The cause isn't the scoring rule this time. It's the training: every practice day began with the battery at 70%, so the AI learned a schedule built on that assumption.
- An AI is only tested on the world it trained in. A real grid never resets at midnight.

**Next:** notebook 06 retrains the agent on continuous weeks instead of single days, so it has to live with the battery it leaves behind, and reruns this test.